In [34]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(12046)

In [35]:
sequence_len = 64
if torch.backends.mps.is_available():
    device = torch.device("mps")
    print("使用MPS！")
else:
    device = torch.device("cpu")
    print("使用CPU")

使用MPS！


In [36]:
# k:(B,T,H) H:特征个数
# Q:(B,T,H)
# k @ Q.transpose(-2,-1) : (B,T,T)

In [37]:
# 创造半三角的例子
scores =  torch.randn(1,4,4)
print(scores)

tensor([[[ 1.0185, -1.3091,  1.2908,  0.5276],
         [-0.2985,  1.6259,  2.0433, -0.6417],
         [ 0.8795, -1.0512,  1.1491,  0.6116],
         [ 0.2128, -0.5512,  0.0450,  0.5010]]])


In [38]:
# 定义下三角矩阵
tril = torch.tril(torch.ones(4,4))
print(tril)

tensor([[1., 0., 0., 0.],
        [1., 1., 0., 0.],
        [1., 1., 1., 0.],
        [1., 1., 1., 1.]])


In [39]:
s = scores.masked_fill(tril ==0, float('-inf'))
print(s) #行的和不一定为 1，代表的是相对重要性的原始度量

tensor([[[ 1.0185,    -inf,    -inf,    -inf],
         [-0.2985,  1.6259,    -inf,    -inf],
         [ 0.8795, -1.0512,  1.1491,    -inf],
         [ 0.2128, -0.5512,  0.0450,  0.5010]]])


In [40]:
# 定义权重分布
F.softmax(s, dim= -1) #每一行的和为 1, 代表的是归一化后的注意力权重

tensor([[[1.0000, 0.0000, 0.0000, 0.0000],
         [0.1274, 0.8726, 0.0000, 0.0000],
         [0.4074, 0.0591, 0.5335, 0.0000],
         [0.2743, 0.1278, 0.2319, 0.3659]]])

In [41]:
# softmax对方差的敏感性
x = torch.randn(1,8)
x.var()

tensor(1.1908)

In [42]:
print(F.softmax(x, dim= -1))

tensor([[0.1236, 0.0400, 0.0791, 0.0175, 0.6045, 0.0262, 0.0633, 0.0457]])


In [43]:
# Softmax 的"锐化"效应，当输入值被放大时，Softmax 的输出分布会变得更加"极端"。
print(F.softmax(x*100, dim= -1))

tensor([[0., 0., 0., 0., 1., 0., 0., 0.]])


In [44]:
# 对齐分数的方差变化
B, T, H = 32, 100, 10
K = torch.randn(B, T, H)
Q = torch.randn(B, T, H)
scores = K @ Q.transpose(-2,-1)/H ** 0.5
scores.var() #保持方差为1避免， Softmax 进入饱和区（梯度消失），保持梯度稳定性， 使训练更稳定

tensor(1.0130)

In [45]:
def attention(query, key, value, dropout, mask=None):
    # query:             (B, T, H)
    # mask:                 (T, T)
    # output:            (B, T, H)
    B, T, H = query.shape
    score = query @ key.transpose(-2,-1) / H ** 0.5
    if mask is not None:
        scores = score.masked_fill(mask == 0, float('-inf'))
    w_att = F.softmax(scores, dim=-1)  # (B, T, T)
    out = w_att @ value                # (B, T, H)
    return out

In [50]:
#实现单向自注意力
class MaskedAttention(nn.Module):
    # emb_size:输入向量长度，head_size：背景向量长度
    """
    RNN：需要偏置来拟合数据中的偏移
    Transformer：偏置会被注意力分数计算"抵消"
    """
    def __init__(self, emb_size, head_size):
        # emb_size: C, head_size: H
        super(MaskedAttention, self).__init__()
        self.key = nn.Linear(emb_size, head_size, bias=False)
        self.query = nn.Linear(emb_size, head_size, bias=False)
        self.value = nn.Linear(emb_size, head_size, bias=False)
        # 定义下三角
        self.register_buffer('trill', torch.tril(torch.ones(sequence_len,sequence_len))) #注册为缓冲区，使mask的执行设备一致
        self.dp = nn.Dropout(0.4)

    def forward(self, x):
        #  x : (B, T, C)
        # out: (B, T, C)
        B, T, C = x.shape
        k = self.key(x)     # (B, T, H)
        q = self.query(x)   # (B, T, H)
        v = self.value(x)   # (B, T, H)
        mask = self.trill[:T, :T]
        out = attention(q,k, v, self.dp, mask) #使用注册的缓冲区
        return out

In [51]:
m = MaskedAttention(3,4)
x = torch.randn(5, 10, 3)
m(x).shape

torch.Size([5, 10, 4])